In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/amirmotefaker/ab-testing-dataset/test_group.csv
/kaggle/input/datasets/amirmotefaker/ab-testing-dataset/control_group.csv


Assumptions taken in the Analysis:
1. The data contains marketing campaign health/details on aggregated level (sum and count) for each day.Both the campaigns were run for same 30 days.
2. Wherever there is null value (say null impressions that has been considered as 0 impresssions for that day).
   

In [2]:
control=pd.read_csv("/kaggle/input/datasets/amirmotefaker/ab-testing-dataset/control_group.csv",sep=";")
control.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0
4,Control Campaign,5.08.2019,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
test = pd.read_csv(
    "/kaggle/input/datasets/amirmotefaker/ab-testing-dataset/test_group.csv",
    sep=";"
)
test.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
0,Test Campaign,1.08.2019,3008,39550,35820,3038,1946,1069,894,255
1,Test Campaign,2.08.2019,2542,100719,91236,4657,2359,1548,879,677
2,Test Campaign,3.08.2019,2365,70263,45198,7885,2572,2367,1268,578
3,Test Campaign,4.08.2019,2710,78451,25937,4216,2216,1437,566,340
4,Test Campaign,5.08.2019,2297,114295,95138,5863,2106,858,956,768


In [4]:
control.columns

Index(['Campaign Name', 'Date', 'Spend [USD]', '# of Impressions', 'Reach',
       '# of Website Clicks', '# of Searches', '# of View Content',
       '# of Add to Cart', '# of Purchase'],
      dtype='object')

In [5]:
test.describe()

,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
mean,2563.066667,74584.800000,53491.566667,6032.333333,2418.966667,1858.000000,881.533333,521.233333
std,348.687681,32121.377422,28795.775752,1708.567263,388.742312,597.654669,347.584248,211.047745
min,1968.000000,22521.000000,10598.000000,3038.000000,1854.000000,858.000000,278.000000,238.000000
25%,2324.500000,47541.250000,31516.250000,4407.000000,2043.000000,1320.000000,582.500000,298.000000
50%,2584.000000,68853.500000,44219.500000,6242.500000,2395.500000,1881.000000,974.000000,500.000000
75%,2836.250000,99500.000000,78778.750000,7604.750000,2801.250000,2412.000000,1148.500000,701.000000
max,3112.000000,133771.000000,109834.000000,8264.000000,2978.000000,2801.000000,1391.000000,890.000000


In [6]:
control.describe()

,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
count,30.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,2288.433333,109559.758621,88844.931034,5320.793103,2221.310345,1943.793103,1300.000000,522.793103
std,367.334451,21688.922908,21832.349595,1757.369003,866.089368,777.545469,407.457973,185.028642
min,1757.000000,71274.000000,42859.000000,2277.000000,1001.000000,848.000000,442.000000,222.000000
25%,1945.500000,92029.000000,74192.000000,4085.000000,1615.000000,1249.000000,930.000000,372.000000
50%,2299.500000,113430.000000,91579.000000,5224.000000,2390.000000,1984.000000,1339.000000,501.000000
75%,2532.000000,121332.000000,102479.000000,6628.000000,2711.000000,2421.000000,1641.000000,670.000000
max,3083.000000,145248.000000,127852.000000,8137.000000,4891.000000,4219.000000,1913.000000,800.000000


In [7]:
#function to print null entries
def check_blanks(df):
    print("Column Name : Count of Blanks")
    for col_name in df.columns:
        count=df[col_name].isnull().sum()
        print(col_name," : ",count)
print("Control Campaign")
check_blanks(control)

#Function to print empty rows
def show_blanks(df):
    blank_row=df.isnull()
    #axis=1 means ki row mei kahi par bhi blank ho to consider karle
    blank_row=df[blank_row.any(axis=1)]
    print(blank_row.head())

print("\nRows with missing values")
show_blanks(control)

Control Campaign
Column Name : Count of Blanks
Campaign Name  :  0
Date  :  0
Spend [USD]  :  0
# of Impressions  :  1
Reach  :  1
# of Website Clicks  :  1
# of Searches  :  1
# of View Content  :  1
# of Add to Cart  :  1
# of Purchase  :  1

Rows with missing values
      Campaign Name       Date  Spend [USD]  # of Impressions  Reach  \
4  Control Campaign  5.08.2019         1835               NaN    NaN   

   # of Website Clicks  # of Searches  # of View Content  # of Add to Cart  \
4                  NaN            NaN                NaN               NaN   

   # of Purchase  
4            NaN  


In [8]:
#similarly we will do the same with test campaign
print("Control Campaign")
check_blanks(test)
print("\nRows with missing values")
show_blanks(test)
#test has no entry with blanks

Control Campaign
Column Name : Count of Blanks
Campaign Name  :  0
Date  :  0
Spend [USD]  :  0
# of Impressions  :  0
Reach  :  0
# of Website Clicks  :  0
# of Searches  :  0
# of View Content  :  0
# of Add to Cart  :  0
# of Purchase  :  0

Rows with missing values
Empty DataFrame
Columns: [Campaign Name, Date, Spend [USD], # of Impressions, Reach, # of Website Clicks, # of Searches, # of View Content, # of Add to Cart, # of Purchase]
Index: []


In [9]:
#funcation for finding duplicate rows 
#since this data is DOD level ... we will check if there are more than 1 entry for same date
def check_duplicates(df,subset_columns):
    duplicates=df.duplicated(subset=subset_columns)
    print("No. of duplicates: ", len(df[duplicates]))
    print("Duplicate Rows :")
    print(df[duplicates])
print("Control Campaign")
check_duplicates(control,"Date")
print("\nTest Campaign")
check_duplicates(test,"Date")
#there are no duplicate entries

Control Campaign
No. of duplicates:  0
Duplicate Rows :
Empty DataFrame
Columns: [Campaign Name, Date, Spend [USD], # of Impressions, Reach, # of Website Clicks, # of Searches, # of View Content, # of Add to Cart, # of Purchase]
Index: []

Test Campaign
No. of duplicates:  0
Duplicate Rows :
Empty DataFrame
Columns: [Campaign Name, Date, Spend [USD], # of Impressions, Reach, # of Website Clicks, # of Searches, # of View Content, # of Add to Cart, # of Purchase]
Index: []


Costing Analysis Metrics:

1. ClickThroughRate (CTR)e: No. of Website Clicks/ No. of Impressions (Kitne log Ad Dekh kar rahe hai)
2. CostperClick (CPC) :  Amount Spend / No of Clicks
3. Cost per Impression (CPI) : Amount Spend / No. of Impressions
4. Customer Aquisition Cost (CAC) : Amount Spend / No of Conversions (Amount Spent on converting each customer)

Website Engagement Metrics
1. Search Rate: No. of searches/ No. of website clicks (Basically a metric on calculating are the users engaging(use the website) post landing on it)
2. Add2Cart_Rate: No. of Add to Cart/ No. of View Content (Basically once the customer has landed on the website - Do they like it, If ATC is low that means the consumers didnot like the product/website or find it very expensive) 
3. Conversion Rate: No of Conersions/ No. of Website Clicks
4. Cart Abandonment Rate: (total add_to_cart-total_purchases)/total add_to cart (Means how many people add the product to cart but dont buy it)


In [10]:
control["CTR"]=control["# of Website Clicks"]*100/control["# of Impressions"]
control["CPC"]=control["Spend [USD]"]/control["# of Website Clicks"]
control["CPI"]=control["Spend [USD]"]/control["# of Impressions"]
control["Customer Aquisition Cost"]=control["Spend [USD]"]/control["# of Purchase"]
#Website Engagement Metrics
control["Search Rate"]=control["# of Searches"]*100/control["# of Website Clicks"]
control["Add2Cart_Rate"]=control["# of Add to Cart"]*100/control["# of Website Clicks"]
control["Conversion Rate"]=control["# of Purchase"]*100/control["# of Website Clicks"]
control["Cart Abandonment Rate"]=100-(control["# of Purchase"]*100/control["# of Add to Cart"])
control.head()


,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase,CTR,CPC,CPI,Customer Aquisition Cost,Search Rate,Add2Cart_Rate,Conversion Rate,Cart Abandonment Rate
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0,8.483471,0.324971,0.027569,3.689320,32.639681,25.926454,8.808438,66.025289
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0,6.700264,0.216646,0.014516,3.438356,25.067818,15.030826,6.300863,58.080394
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0,4.941121,0.360018,0.017789,6.298387,26.690227,17.424708,5.716042,67.195767
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0,4.205659,0.632953,0.026620,5.705882,33.996737,38.597064,11.092985,71.259510
4,Control Campaign,5.08.2019,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
#kyuki code mei NaN values / Inf values aa sakti hai to usko solve karne ke liye 
control.replace([np.inf,-np.inf],0,inplace=True)
control.fillna(0,inplace=True)
control.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase,CTR,CPC,CPI,Customer Aquisition Cost,Search Rate,Add2Cart_Rate,Conversion Rate,Cart Abandonment Rate
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0,8.483471,0.324971,0.027569,3.689320,32.639681,25.926454,8.808438,66.025289
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0,6.700264,0.216646,0.014516,3.438356,25.067818,15.030826,6.300863,58.080394
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0,4.941121,0.360018,0.017789,6.298387,26.690227,17.424708,5.716042,67.195767
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0,4.205659,0.632953,0.026620,5.705882,33.996737,38.597064,11.092985,71.259510
4,Control Campaign,5.08.2019,1835,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [12]:
#lets make these metrics for test campiagn
test["CTR"]=test["# of Website Clicks"]*100/test["# of Impressions"]
test["CPC"]=test["Spend [USD]"]/test["# of Website Clicks"]
test["CPI"]=test["Spend [USD]"]/test["# of Impressions"]
test["Customer Aquisition Cost"]=test["Spend [USD]"]/test["# of Purchase"]
#Website Engagement Metrics
test["Search Rate"]=test["# of Searches"]*100/test["# of Website Clicks"]
test["Add2Cart_Rate"]=test["# of Add to Cart"]*100/test["# of Website Clicks"]
test["Conversion Rate"]=test["# of Purchase"]*100/test["# of Website Clicks"]
test["Cart Abandonment Rate"]=100-(test["# of Purchase"]*100/test["# of Add to Cart"])
test.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase,CTR,CPC,CPI,Customer Aquisition Cost,Search Rate,Add2Cart_Rate,Conversion Rate,Cart Abandonment Rate
0,Test Campaign,1.08.2019,3008,39550,35820,3038,1946,1069,894,255,7.681416,0.990125,0.076056,11.796078,64.055300,29.427255,8.393680,71.476510
1,Test Campaign,2.08.2019,2542,100719,91236,4657,2359,1548,879,677,4.623755,0.545845,0.025239,3.754801,50.654928,18.874812,14.537256,22.980660
2,Test Campaign,3.08.2019,2365,70263,45198,7885,2572,2367,1268,578,11.222123,0.299937,0.033659,4.091696,32.618897,16.081167,7.330374,54.416404
3,Test Campaign,4.08.2019,2710,78451,25937,4216,2216,1437,566,340,5.374055,0.642789,0.034544,7.970588,52.561670,13.425047,8.064516,39.929329
4,Test Campaign,5.08.2019,2297,114295,95138,5863,2106,858,956,768,5.129708,0.391779,0.020097,2.990885,35.920177,16.305646,13.099096,19.665272


In [13]:
#kyuki code mei NaN values / Inf values aa sakti hai to usko solve karne ke liye 
test.replace([np.inf,-np.inf],0,inplace=True)
test.fillna(0,inplace=True)
test.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase,CTR,CPC,CPI,Customer Aquisition Cost,Search Rate,Add2Cart_Rate,Conversion Rate,Cart Abandonment Rate
0,Test Campaign,1.08.2019,3008,39550,35820,3038,1946,1069,894,255,7.681416,0.990125,0.076056,11.796078,64.055300,29.427255,8.393680,71.476510
1,Test Campaign,2.08.2019,2542,100719,91236,4657,2359,1548,879,677,4.623755,0.545845,0.025239,3.754801,50.654928,18.874812,14.537256,22.980660
2,Test Campaign,3.08.2019,2365,70263,45198,7885,2572,2367,1268,578,11.222123,0.299937,0.033659,4.091696,32.618897,16.081167,7.330374,54.416404
3,Test Campaign,4.08.2019,2710,78451,25937,4216,2216,1437,566,340,5.374055,0.642789,0.034544,7.970588,52.561670,13.425047,8.064516,39.929329
4,Test Campaign,5.08.2019,2297,114295,95138,5863,2106,858,956,768,5.129708,0.391779,0.020097,2.990885,35.920177,16.305646,13.099096,19.665272
